In [8]:
import kagglehub
import os

path = kagglehub.dataset_download(
    "thedevastator/open-orca-augmented-flan-dataset"
)

print(path)

for file in os.listdir(path):
    print(file)

Mounting files to /kaggle/input/datasets/thedevastator/open-orca-augmented-flan-dataset...
/kaggle/input/datasets/thedevastator/open-orca-augmented-flan-dataset
train.csv


In [17]:
from tqdm import tqdm
import pandas as pd

tqdm.pandas()


# Load dataset
df = pd.read_csv(
    "/kaggle/input/datasets/thedevastator/open-orca-augmented-flan-dataset/train.csv"
)


print("="*50)
print("Dataset Loaded")
print("="*50)

print(df.head())
print("\nColumns:")
print(df.columns)


# Example: add tqdm during processing
# Replace column names after checking df.columns



Dataset Loaded
             id                                      system_prompt  \
0    niv.242684                                                NaN   
1   flan.564327  You are an AI assistant. You will be given a t...   
2  flan.1875913  You are a helpful assistant, who always provid...   
3     t0.408370  You are an AI assistant. You will be given a t...   
4     cot.86217  You are an AI assistant that helps people find...   

                                            question  \
0  You will be given a definition of a task first...   
1  Generate an approximately fifteen-word sentenc...   
2  What happens next in this paragraph?\n\nShe th...   
3  Please answer the following question: I want t...   
4  James runs a TV show and there are 5 main char...   

                                            response  
0  [\n  ["AFC Ajax (amateurs)", "has ground", "Sp...  
1  Midsummer House is a moderately priced Chinese...  
2  C. She then dips the needle in ink and using t...  
3  Base

In [18]:
df.columns

Index(['id', 'system_prompt', 'question', 'response'], dtype='object')

In [21]:
pd.set_option("display.max_colwidth", None)

In [31]:
df.sample(5)[['question','response']]

,question,response
2220127,"Please answer the following question: A few times. Mostly whilst doing this job and trying to help drunk people or assholes. Sadly it's becoming a regular occurrence for crews to be assaulted. My last time was a few days ago and the worst thing about it was watching my crew mate and several police officers curl up laughing. I'll set the scene. Imagine. If you will. A large rather expensive hotel. Then add to that image a rather drunk and buxom young lady who is lying on the floor in a drunken slumber after taking off a lot of her clothes. She was just in her underwear. We were called to look after her. On our arrival she was absolutely fine. Stood up, walked to the stretcher and lay down. Whining about how we were the ' oppressors of society', and other such drivel Because I had the cheek to try and give her a tiny bit of dignity by covering her up with a blanket. She didn't want that. We made it to the ambulance without any hassle and started to do the necessary checks we do on every patient. Blood pressure etc etc when out of the blue she decided that she wanted to release her rather large breasts. Try as I might to give her a little bit of dignity she wasn't interested so we just made sure no one could see inside the ambulance. After a few minutes she calmed down so I put the blanket over her. Something I do for almost every patient. Without warning THWACK. she hit my in the face with her boob. Then. As I reeled from this waiting for my brain to catch up and make sense of everything, THWACK she did it again. Right round the face with a large breast!! I turned around to get some support from my colleagues who were crying. Both my crew mate and the officer. Crying. Neither able to speak……….. I didn't know what to say for a while after that…something my crew mate called 'a blessing' Sigh When were the crew laughing? Pick the correct answer from the following options: A. not enough information B. After the woman's dramatic episode C. Before arriving at the hotel D. Before the woman's dramatic episode\nA:","Based on the information provided, it is not clear when exactly the crew was laughing. The user mentioned that their crew mate and several police officers were laughing after the incident with the drunk woman, but it is not specified whether this laughter occurred before or after the woman's dramatic episode. Therefore, the correct answer would be A. not enough information."
1245099,"Given the question: W: Must you? It's still early. M: I'm afraid I really must. I have to meet a midnight plane. W: In that case, we can't keep you. M: Well, thank you very much for a pleasant evening. W: Thank you for coming. You must come again. M: Thank you. I will. Good-bye. W: Good-bye. I wish you a very good journey home. What was said before this conversation?\nThe answer is:",The question does not provide any context or information about what was said before this conversation.
1799799,"Read the following article and select the best answer. Article: Hong Kong, a city of seven million--and growing fast. By the year 2050 Hong Kong's population will almost double. Where will these people live? There's only one place to go--up. Fast forward 50 years.This is the tallest man-made structure on the planet. Its name:Millennium Tower. It would stand twice as tall as anything built before. Over 60 thousand people would live or work here. Millennium's designers think it represents the best solution for the coming population explosion. It would cost ten billion dollars and consume more building materials than any single nation could produce. David Nelson, one of Millennium's key designers, said:""It would be a massive project and a world project. But to actually realize it, resources would have to come from all over the globe. The challenge and the motivation for doing it would be to realize one of the world's largest and most complex construction projects."" Millennium would be more complex than anything ever built. So its 

In [33]:
# Remove empty values
df = df.dropna(
    subset=["question", "response"]
)


In [34]:
def count_words(text):
    return len(str(text).split())

In [36]:
df["prompt_words"] = df["question"].progress_apply(
    count_words
)

df["response_words"] = df["response"].progress_apply(
    count_words
)

100%|██████████| 4233754/4233754 [00:57<00:00, 73422.62it/s]
/tmp/ipykernel_58/1462864720.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["prompt_words"] = df["question"].progress_apply(
100%|██████████| 4233754/4233754 [00:28<00:00, 146460.08it/s]
/tmp/ipykernel_58/1462864720.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["response_words"] = df["response"].progress_apply(


In [37]:
print("Before filtering:")
print(len(df))

Before filtering:
4233754


In [38]:
# Apply constraints

filtered_df = df[
    (df["prompt_words"] >= 10) &
    (df["prompt_words"] <= 120) &
    (df["response_words"] >= 20) &
    (df["response_words"] <= 250)
]

In [39]:
print("\nAfter filtering:")
print(len(filtered_df))


After filtering:
1784519


In [43]:
# Check examples

print("\nExample:")
print(filtered_df.iloc[10]["question"])
print("\nResponse:")
print(filtered_df.iloc[10]["response"])


Example:
If "An older couple with joke glasses and cigars." does that mean that "A couple looking silly."?
Options:
- yes
- it is not possible to tell
- no Let's be accurate as possible and think first.

Response:
Yes, "An older couple with joke glasses and cigars" would generally imply "A couple looking silly," as joke glasses and cigars are items typically used to create a humorous or lighthearted appearance.


In [45]:
virgo_df = filtered_df.rename(
    columns={
        "question": "prompt",
        "response": "response"
    }
)


virgo_df = virgo_df[
    [
        "prompt",
        "response"
    ]
]

In [50]:
virgo_df.sample(5)

,prompt,response
3123058,"even if you are not satisfied, you can contact us and we will try to resolve your problem as soon as possible.\n\nPlease write the text above using proper case.","Even if you are not satisfied, you can contact us and we will try to resolve your problem as soon as possible."
3570223,Answer the question below: When water freezes into ice it becomes less dense. or more dense? Assuming that: Water becomes more dense when it is colder and when it has more salt.\nA:,"When water freezes into ice, it becomes less dense. This is because the molecules in water are more spread out in ice than in liquid water. As a result, ice floats on top of liquid water. This property of water is important for aquatic life, as it allows fish and other organisms to survive in cold temperatures by staying in the warmer water below the ice. The assumption that water becomes more dense when it is colder and when it has more salt is correct, but it does not apply to the case of water freezing into ice."
2597356,"Answer the following question: Answer the question based on the following text. Question: Micah is entering the atmosphere, and is approaching the Earth's surface. As he's passing through the troposphere, the air is going to get warmer or colder? Text: Air in the troposphere is warmer closer to Earths surface.\nAnswer:","As Micah is entering the atmosphere and approaching the Earth's surface, he will be passing through the troposphere. According to the given text, the air in the troposphere is warmer closer to Earth's surface. Therefore, as Micah passes through the troposphere, the air around him will get warmer."
808163,"Can we conclude from ""People standing around taking video."" that ""No one is taking a video.""?\nOptions:\n- yes\n- no\n- it is not possible to tell Let's be accurate as possible and think first.","No, we cannot conclude from ""People standing around taking video."" that ""No one is taking a video."" In fact, the statement clearly indicates that there are people who are taking a video, so the conclusion should be the opposite: people are taking videos."
3237077,"Can we draw the following hypothesis from the context (see options)? \n\nContext:\n\nHow to survive a boring party<br>Keep an open mind. Although you may be feeling bored, keeping an open mind and attitude can be helpful. Being open to conversation and other party guests will help you stay engaged and make the party seem a lot more bearable.\n\nHypothesis: Help yourself stay engaged and make the party more bearable by staying on the phone.\nChoose your answer from: --Yes; --It's impossible to say; --No;","No, we cannot draw the hypothesis ""Help yourself stay engaged and make the party more bearable by staying on the phone"" from the given context. The context suggests that keeping an open mind and engaging in conversation with other party guests can help make the party more bearable. There is no mention of using a phone to stay engaged."


In [51]:
def normalize_text(text):

    text = str(text).strip()

    # Normalize line endings
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Replace newline characters with Virgo token
    text = text.replace("\n", "<newline>")

    # Remove repeated newline tokens
    while "<newline><newline><newline>" in text:
        text = text.replace(
            "<newline><newline><newline>",
            "<newline><newline>"
        )

    return text

In [53]:
from tqdm import tqdm

tqdm.pandas()


virgo_df["prompt"] = virgo_df["prompt"].progress_apply(
    normalize_text
)

virgo_df["response"] = virgo_df["response"].progress_apply(
    normalize_text
)

100%|██████████| 1784519/1784519 [00:05<00:00, 342432.73it/s]


In [54]:
virgo_df.sample(5)

,prompt,response
2646390,"Leo: Premise: ""A young man rides a bucking horse as several others in a crowd look on.""<newline>Hypothesis: ""A man in a rodeo.""<newline>Is the hypothesis entailed by the premise?<newline>Options:<newline>- yes<newline>- it is not possible to tell<newline>- no<newline>Mei: OK, So, let's think first...<newline>Me:","Based on the given premise, it is likely that the young man is participating in a rodeo event where he is riding a bucking horse. Therefore, the hypothesis ""A man in a rodeo"" is entailed by the premise. So, the answer is ""yes""."
735114,"Translate """"The way the market is developing, if we speak again in a year, we will have 20 more such references to add to the list,"" said Rigal."" to French?","""La manière dont le marché évolue, si nous parlons à nouveau dans un an, nous aurons 20 références de plus à ajouter à la liste"", a déclaré Rigal.<newline><newline>Now, let me explain this to you like you're five years old: This person called Rigal is talking about something called ""the market"" which is like a big place where people buy and sell things. Right now, the market is growing and changing. Rigal thinks that if they talk again in one year (which is as long as it takes for your birthday to come back around), they will have 20 more things to talk about and add to a special list they have."
2463382,"Q:The exercise is to decide whether the question accepts the proposed suggestion as a correct answer. If yes, write ""True"", otherwise write ""False"". Question: when was coca cola invented Suggestion: The Coca-Cola Company has, on occasion, introduced other cola drinks under the Coke brand name.<newline>A:",False. The suggestion does not answer the question. The correct answer to the question is that Coca-Cola was invented in 1886.
884128,Q:What's the answer to that question: who are the players on the toronto maple leafs?<newline>A:,"As an AI, I can provide you with the most current available information. The roster of the Toronto Maple Leafs can change regularly due to trades and roster updates. Here are some key players from their current roster, as of October 2021:<newline><newline>1. Auston Matthews<newline>2. John Tavares<newline>3. Mitch Marner<newline>4. William Nylander<newline>5. Morgan Rielly<newline>6. T.J. Brodie<newline>7. Jake Muzzin<newline>8. Michael Bunting<newline>9. Alexander Kerfoot<newline>10. Ilya Mikheyev<newline><newline>Please note that the roster may change over time, and for the most up-to-date information, you can visit the official Toronto Maple Leafs website or follow their social media channels."
3779606,"Given the sentence ""A football punter with a white and red jersey who is about to punt a kick and his teammates are giving chase after the ball."" is it true that ""A professional football punter attemps an onside kick as others look on from the stands at a football game.""? Step by step answer:","No, it is not true that ""A professional football punter attempts an onside kick as others look on from the stands at a football game."" based on the given sentence ""A football punter with a white and red jersey who is about to punt a kick and his teammates are giving chase after the ball."" Here's a step-by-step explanation:<newline><newline>1. The given sentence describes a football punter wearing a white and red jersey.<newline>2. The punter is about to punt a kick, which means he is preparing to kick the ball.<newline>3. The punter's teammates are giving chase after the ball, which means they are running after the ball that the punter is about to kick.<newline>4. There is no mention of an onside kick in the given sentence.<newline>5. The sentence does not mention anything about people looking on from the stands at a football game.<newline><newline>Therefore, based on the given sentence, it is not true that ""A professional football punter attempts an onside kick as others look on from the stands at a football game."""


In [55]:
!pip install langdetect -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 26.9 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done


In [57]:
import re
from tqdm import tqdm

tqdm.pandas()


def is_english_only(text):
    text = str(text)

    # Allow:
    # A-Z a-z
    # numbers
    # punctuation
    # common symbols
    pattern = r"^[\x00-\x7F]+$"

    return bool(re.match(pattern, text))


# Check instruction and response

virgo_df["prompt_english"] = virgo_df["prompt"].progress_apply(
    is_english_only
)

virgo_df["response_english"] = virgo_df["response"].progress_apply(
    is_english_only
)


# Keep only fully English rows

virgo_df = virgo_df[
    (virgo_df["prompt_english"]) &
    (virgo_df["response_english"])
]


print("Remaining samples:", len(df))

100%|██████████| 1784519/1784519 [00:07<00:00, 224313.37it/s]


Remaining samples: 4233754


In [60]:
virgo_df.sample(10)

,prompt,response,prompt_english,response_english
3698033,"Please answer the following question: Process: - Carbon dioxide passes through stomata in the leaves - Water is absorbed through roots - Chloroplast cells in leaves contain chlorophyll - Chlorophyll absorbs energy from sunlight - Energy splits water molecules into hydrogen and oxygen - Oxygen gets released into atmosphere - Hydrogen and carbon dioxide used to create glucose - Glucose is food for plants. suppose The air is pollution free happens, how will it affect MORE sugar and oxygen being produced. Which of the following is the supposed perturbation? - directly impacting a step of the process - indirectly impacting a step of the process - not impacting any step of the process<newline>A:","If the air is pollution-free, it will directly impact the step of the process where carbon dioxide passes through stomata in the leaves. With less pollution, there will be more carbon dioxide available for the process of photosynthesis, leading to more sugar and oxygen being produced. Therefore, the supposed perturbation is directly impacting a step of the process.",True,True
2662769,"Read the following article and answer the question.<newline><newline>This is normal ! We ca n't always have the smooch we want when we want it . We are both stressed about money and work ( lack of both ) at the moment . A is not by nature a worrier , but she 's worrying now .<newline><newline>What may happen to their relationship during this situation ?","Based on the given information, it is likely that the couple's relationship may experience some strain or tension due to their current stress about money and work. The fact that the person referred to as ""A"" is not typically a worrier but is now worrying suggests that the situation is causing some level of distress. This could potentially lead to arguments or disagreements between the couple, as well as a decrease in intimacy or affection (as indicated by the mention of not being able to have the ""smooch"" they want). However, without further information it is difficult to predict the exact outcome of their relationship during this situation.",True,True
3495039,"Q:Formulate an answer to this elaborate question: Shownu, Kihyun, Hyungwon, Wonho, Minhyuk, and I.M. and what other South Korean comprise a group created in 2015?<newline>A:","The group that was created in 2015 and includes Shownu, Kihyun, Hyungwon, Wonho, Minhyuk, and I.M. is none other than Monsta X. Monsta X is a South Korean boy band that was formed by Starship Entertainment through the survival show ""No.Mercy"". The group debuted on May 14, 2015, with their first extended play ""Trespass"". <newline><newline>Monsta X has gained a significant following both domestically and internationally, thanks to their unique blend of hip-hop, pop, and electronic music. The group has released several successful albums and singles, including ""Alligator"", ""Shoot Out"", and ""Love Killa"". <newline><newline>In addition to their music, Monsta X has also made a name for themselves through their energetic and captivating live performances. The group has embarked on several world tours, including their ""We Are Here"" tour in 2019, which saw them perform in cities across Asia, Europe, and North America. <newline><newline>Overall, Monsta X has become one of the most popular and successful K-pop groups in recent years, thanks to their talent, hard work, and dedication to their craft. Their unique sound and style have won them a legion of fans around the world, and they continue to push the boundaries of what is possible in the world of K-pop.",True,True
1348885,Q:FINAL EXAM Question 1. Are both Limp Bizkit and Third Eye Blind rock bands?<newline>A:,"Yes, both Limp Bizkit and Third Eye Blind are considered rock bands. Limp Bizkit is known for their nu-metal sound, while Third Eye Blind is known for their alternative rock sound.",True,True
3408094,Taylor started toward Robin and then asked Robin to go to t

In [62]:
def replace_tabs(text):
    if text is None:
        return ""

    return str(text).replace("\t", "<tab>")


virgo_df["prompt"] = virgo_df["prompt"].apply(
    replace_tabs
)

virgo_df["response"] = virgo_df["response"].apply(
    replace_tabs
)

In [69]:
virgo_df.sample(4)

,prompt,response,prompt_english,response_english
2906356,"Premise: ""Novak attacks the president for his tactics of delay."" If this premise is true, what does that tell us about whether it entails the hypothesis ""Novak loves all of the president's decisions and fully supports them.""?","If the premise is true, it tells us that Novak disagrees with the president's tactics of delay. However, it does not necessarily mean that Novak loves all of the president's decisions and fully supports them. Disagreeing with one decision or tactic does not necessarily mean disagreeing with all decisions or tactics.",True,True
4126168,"Q:Question: Where would you put a board game after you finish using it? Choices: - den - shelf - house - toy store - closet The rationale to choose ""shelf"" as the answer is that:<newline>A:","The most logical place to put a board game after using it would be on a shelf. This way, it can be easily accessed for future use and kept organized with other board games or similar items. The other options, such as the den, house, toy store, or closet, may not be as practical or convenient for storing board games.",True,True
329202,"Here is a dialogue:<newline>Lia: Are you going to the party tonight?<newline>Brian: I have a headache...<newline>Lia: Take a painkiller <newline>Brian: and then drink? I am not sure about the brightness of this idea.<newline>Lia: Hahaha, so you won't go. Pity.<newline>Brian: Next time!<newline><newline>Write a short summary!","In this dialogue, Lia asks Brian if he's attending the party tonight, to which he hesitates, mentioning his headache. Lia suggests taking a painkiller, but Brian isn't keen on combining it with alcohol. They laugh, and Brian turns down the invitation, promising to attend next time.",True,True
2347414,"Given the question: Here's a question and a few possible answers: Q: What area is likely to have at least one concert hall? Possible A: commercial building, muscians, theater, university building, downtown Why is ""university building"" an answer aligned with human common sense?<newline>The answer is:","""University building"" is an answer aligned with human common sense because universities often have a strong focus on the arts and culture. Many universities have music, theater, and dance programs, which require performance spaces such as concert halls. Additionally, universities often host concerts and other cultural events for the surrounding community, making a concert hall a valuable addition to a university building. Furthermore, universities are often located in urban areas, which may have a high demand for cultural events and venues. Therefore, it is not uncommon for universities to have concert halls or other performance spaces within their buildings. Overall, the presence of a concert hall in a university building aligns with the idea that universities are not only centers of academic learning, but also hubs of cultural activity.",True,True


In [70]:
from tqdm import tqdm

tqdm.pandas()


def format_virgo(row):

    return (
        "<bos>"
        "User: "
        + row["prompt"]
        + "<newline><newline>"
        "Assistant: "
        + row["response"]
        + "<eos>"
    )


virgo_df["text"] = virgo_df.progress_apply(
    format_virgo,
    axis=1
)

100%|██████████| 1461532/1461532 [00:15<00:00, 94454.68it/s] 


In [71]:
virgo_df.sample(3)

,prompt,response,prompt_english,response_english,text
2523690,Choose from: 1). No 2). Yes<newline>Q: Is this product review positive? Title: Great film!!!!! Review: This movie is definitely one to add to your collection. It is watchable over and over and endlessly entertaining!! Great cast!!! I hope they do a sequel! brillant! Answer:<newline>A:,"2) Yes, the product review is positive. <newline><newline>Justification: <newline>1. The title of the review is ""Great film!!!!!"" which indicates a positive sentiment towards the product.<newline>2. The reviewer mentions that the movie is ""watchable over and over and endlessly entertaining!!"" which suggests that they enjoyed the movie and found it entertaining.<newline>3. The reviewer also praises the cast by saying ""Great cast!!!"" which further indicates a positive sentiment towards the product.<newline>4. The reviewer expresses their desire for a sequel by saying ""I hope they do a sequel!"" which shows that they enjoyed the movie and would like to see more of it. <newline><newline>Based on these reasons, it can be concluded that the product review is positive.",True,True,"<bos>User: Choose from: 1). No 2). Yes<newline>Q: Is this product review positive? Title: Great film!!!!! Review: This movie is definitely one to add to your collection. It is watchable over and over and endlessly entertaining!! Great cast!!! I hope they do a sequel! brillant! Answer:<newline>A:<newline><newline>Assistant: 2) Yes, the product review is positive. <newline><newline>Justification: <newline>1. The title of the review is ""Great film!!!!!"" which indicates a positive sentiment towards the product.<newline>2. The reviewer mentions that the movie is ""watchable over and over and endlessly entertaining!!"" which suggests that they enjoyed the movie and found it entertaining.<newline>3. The reviewer also praises the cast by saying ""Great cast!!!"" which further indicates a positive sentiment towards the product.<newline>4. The reviewer expresses their desire for a sequel by saying ""I hope they do a sequel!"" which shows that they enjoyed the movie and would like to see more of it. <newline><newline>Based on these reasons, it can be concluded that the product review is positive.<eos>"
2760537,"Given the question: Question: ""Where can memory cells be read quicker"" Context: ""The CPU contains a special set of memory cells called registers that can be read and written to much more rapidly than the main memory area. There are typically between two and one hundred registers depending on the type of CPU. Registers are used for the most frequently needed data items to avoid having to access main memory every time data is needed. As data is constantly being worked on, reducing the need to access main memory (which is often slow compared to the ALU and control units) greatly increases the computer's speed."" Answer:<newline>The answer is:","Memory cells can be read quicker in the CPU's special set of memory cells called registers. Registers can be read and written to much more rapidly than the main memory area. Registers are used for the most frequently needed data items to avoid having to access main memory every time data is needed. This greatly increases the computer's speed by reducing the need to access main memory, which is often slow compared to the ALU and control units. The number of registers in a CPU typically ranges from two to one hundred, depending on the type of CPU.",True,True,"<bos>User: Given the question: Question: ""Where can memory cells be read quicker"" Context: ""The CPU contains a special set of memory cells called registers that can be read and written to much more rapidly than the main memory area. There are typically between two and one hundred registers depending on the type of CPU. Registers are used for the most frequently needed data items to avoid having to access main memory every time data is needed. As data is constantly being worked on, reducing the need to access main 

In [72]:
virgo_df.drop(['text'], axis=1,inplace=True)

In [74]:
virgo_df.sample(3)

,prompt,response,prompt_english,response_english
228281,Generate a correctly punctuated version of the following text: Thyroid replacement may be recommended in this case,"Thyroid replacement may be recommended in this case.<newline><newline>Justification:<newline>1. The original text has a clear sentence structure which conveys a complete thought.<newline>2. The sentence begins with a capital letter.<newline>3. A period is already present at the end of the sentence.<newline><newline>The text is already well punctuated, so no changes are needed.",True,True
1247837,"Here's a question: What is a yard usually part of? Here are possible answers to this question: - subdivision - neighborhood - file - city - property I believe the correct choice is ""property"", here's why:<newline>The answer to this question is:","property. <newline><newline>A yard is typically a part of a property, which refers to a piece of land that is owned by an individual or entity. A property can include a house, a building, or simply an empty lot of land. The yard is the outdoor area surrounding the property, which may be used for various purposes such as gardening, recreation, or parking. While a yard may also be part of a neighborhood or subdivision, the most accurate answer to this question is ""property"".",True,True
2104033,Question 1: Where can I found modern colours and textures for floor tiles in Sydney?<newline>Question 2: Where can I found a leading manufacturer and supplier of ceramic porcelain tiles in Sydney?<newline>Possible answers: * no; * yes;<newline>Are questions 1 and 2 asking the same thing?,"No, questions 1 and 2 are asking about different things. <newline><newline>Question 1 is asking where to find modern colors and textures for floor tiles in Sydney. This means the person is looking for a store or supplier that has a variety of contemporary options for floor tiles.<newline><newline>Question 2 is asking where to find a leading manufacturer and supplier of ceramic porcelain tiles in Sydney. This means the person is looking for a specific company that is known for producing and selling high-quality ceramic porcelain tiles.",True,True


In [75]:
from tqdm import tqdm

tqdm.pandas()


def format_virgo(row):

    return (
        "<bos>"
        "<newline>"
        "User"
        "<newline>"
        + row["prompt"]
        + "<newline><newline>"
        "Virgo"
        "<newline>"
        + row["response"]
        + "<eos>"
    )


virgo_df["text"] = virgo_df.progress_apply(
    format_virgo,
    axis=1
)

100%|██████████| 1461532/1461532 [00:14<00:00, 100696.11it/s]


In [77]:
virgo_df.sample(3)

,prompt,response,prompt_english,response_english,text
1320305,"Please capitalize where necessary: unsure about which type of m21 file you have? do you want to know exactly what it is, who created it, and how to open it?","Sure! Here's how I would complete this task:<newline><newline>Step 1: Identify the task - The user wants me to capitalize the necessary words in a sentence.<newline><newline>Step 2: Read the sentence - ""unsure about which type of m21 file you have? do you want to know exactly what it is, who created it, and how to open it?""<newline><newline>Step 3: Capitalize the necessary words - ""Unsure about which type of M21 file you have? Do you want to know exactly what it is, who created it, and how to open it?""<newline><newline>Step 4: Justify the capitalization - I capitalized the first letter of the first word in each sentence, as well as the first letter of ""M21"" since it appears to be a specific file type. I also capitalized ""Do"" in the second sentence to indicate the start of a new question.<newline><newline>Step 5: Present the completed task - The sentence has been capitalized as requested.",True,True,"<bos><newline>User<newline>Please capitalize where necessary: unsure about which type of m21 file you have? do you want to know exactly what it is, who created it, and how to open it?<newline><newline>Virgo<newline>Sure! Here's how I would complete this task:<newline><newline>Step 1: Identify the task - The user wants me to capitalize the necessary words in a sentence.<newline><newline>Step 2: Read the sentence - ""unsure about which type of m21 file you have? do you want to know exactly what it is, who created it, and how to open it?""<newline><newline>Step 3: Capitalize the necessary words - ""Unsure about which type of M21 file you have? Do you want to know exactly what it is, who created it, and how to open it?""<newline><newline>Step 4: Justify the capitalization - I capitalized the first letter of the first word in each sentence, as well as the first letter of ""M21"" since it appears to be a specific file type. I also capitalized ""Do"" in the second sentence to indicate the start of a new question.<newline><newline>Step 5: Present the completed task - The sentence has been capitalized as requested.<eos>"
1848990,"""poor acting ""<newline>How would the sentiment of this sentence be perceived?","The sentiment of the sentence ""poor acting"" would generally be perceived as negative. The phrase ""poor acting"" is often used to describe a performance that is lacking in skill, emotion, or believability. It suggests that the actor did not do a good job in portraying their character or delivering their lines. <newline><newline>When someone uses the phrase ""poor acting,"" they are usually expressing disappointment or dissatisfaction with the performance they witnessed. This sentiment can be particularly strong if the person had high expectations for the actor or the production. <newline><newline>Overall, the sentiment of ""poor acting"" is negative and implies that the performance was not up to par. It can be a harsh criticism, but it is also a common way for people to express their disappointment with a performance.",True,True,"<bos><newline>User<newline>""poor acting ""<newline>How would the sentiment of this sentence be perceived?<newline><newline>Virgo<newline>The sentiment of the sentence ""poor acting"" would generally be perceived as negative. The phrase ""poor acting"" is often used to describe a performance that is lacking in skill, emotion, or believability. It suggests that the actor did not do a good job in portraying their character or delivering their lines. <newline><newline>When someone uses the phrase ""poor acting,"" they are usually expressing disappointment or dissatisfaction with the performance they witnessed. This sentiment can be particularly strong if the person had high expectations for the actor or the production. <newline><newline>Overall, the sentiment of ""poor acting"" is neg

In [78]:
virgo_df.shape

(1461532, 5)

In [79]:
from tokenizers import Tokenizer

In [81]:
tokenizer_path = '/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json'
tokenizer = Tokenizer.from_file(tokenizer_path)

In [85]:
sample_text = virgo_df.sample(1)["text"].iloc[0]

# print(sample_text)

encoded = tokenizer.encode(sample_text)

print(encoded.tokens)

['<bos>', '<newline>', 'Us', 'er', '<newline>', 'How', 'Ġmight', 'Ġone', 'Ġdescribe', 'Ġthe', 'Ġsentiment', 'Ġof', 'Ġthis', 'Ġreview', '?', '<newline>', 'My', 'Ġchildren', 'Ġcan', 'Ġbe', 'Ġvery', 'Ġshy', 'Ġbut', 'ĠCraig', 'Ġknew', 'Ġexactly', 'Ġhow', 'Ġto', 'Ġmake', 'Ġthem', 'Ġfeel', 'Ġcomfortable', 'Ġand', 'Ġthey', 'Ġactually', 'Ġenjoyed', 'Ġtheir', 'Ġha', 'irc', 'uts', '!', 'Ġ', 'ĠFor', 'Ġa', 'Ġgreat', 'Ġcut', 'Ġand', 'Ġsupreme', 'Ġcustomer', 'Ġservice', 'ĠI', 'Ġdefinitely', 'Ġrecommend', 'ĠCraig', 'Ġas', 'Ġa', 'Ġstyl', 'ist', '.', 'Ġ', 'ĠHe', 'Ġhas', 'Ġbeen', 'Ġmy', 'Ġhair', 'Ġconsultant', 'Ġfor', 'Ġmany', 'Ġyears', '!', '...', 'Cho', 'ose', 'Ġfrom', ':', '<newline>', 'Ġ+', 'negative', ';', '<newline>', 'Ġ+', 'positive', ';', 'ĠI', 'Ġthink', 'Ġthe', 'Ġanswer', 'Ġis', '<newline>', '<newline>', 'V', 'ir', 'go', '<newline>', '+', 'positive', '<newline>', '<newline>', 'The', 'Ġsentiment', 'Ġof', 'Ġthis', 'Ġreview', 'Ġis', 'Ġpositive', '.', 'ĠThis', 'Ġis', 'Ġbecause', 'Ġthe', 'Ġperson', 

In [86]:
from tqdm import tqdm

tqdm.pandas()


def tokenize_text(text):

    return tokenizer.encode(
        text
    ).ids


virgo_df["input_ids"] = virgo_df["text"].progress_apply(
    tokenize_text
)

100%|██████████| 1461532/1461532 [23:36<00:00, 1032.01it/s]


In [87]:
virgo_df["length"] = virgo_df["input_ids"].apply(
    len
)


print(virgo_df["length"].describe())

count    1.461532e+06
mean     1.978479e+02
std      8.439364e+01
min      4.500000e+01
25%      1.360000e+02
50%      1.830000e+02
75%      2.470000e+02
max      4.077000e+03
Name: length, dtype: float64


In [88]:
virgo_df = virgo_df[
    virgo_df["length"] <= 1024
].reset_index(drop=True)

In [89]:
print(
    "Final samples:",
    len(virgo_df)
)

Final samples: 1461523


In [90]:
virgo_dataset = virgo_df[
    [
        "input_ids"
    ]
]

In [91]:
import torch
from tqdm import tqdm


# collect token ids

all_tokens = [
    ids
    for ids in virgo_df["input_ids"]
]


# save as binary

torch.save(
    all_tokens,
    "virgo_chat_v1_tokenized.bin"
)


print("Saved successfully!")

Saved successfully!


In [96]:
torch.save(
    all_tokens,
    "virgo_chat_v1_tokenized.bin"
)

In [97]:
total_tokens = sum(
    len(ids)
    for ids in all_tokens
)

print("Total tokens:", total_tokens)

Total tokens: 289145616
